<h1 style = "color : #0EE071; text-align : center;"><em>Nata Project</em> - Final Notebook</h1>
<p style = "font-size : 16px; text-align: center;">This notebook runs directly from the first, and will be used to create the model.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Machine Learning I</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes, Gustavo Franco & Lucas Casimiro</p>
<br>

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.preprocessing import RobustScaler

import scipy.stats as stats
from scipy.stats import chi2_contingency

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

from sklearn.linear_model import LassoCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [48]:
df_learn = pd.read_csv('Nata_files/learn.csv')
df_predict = pd.read_csv('Nata_files/predict.csv')

Data Preprocessing

In [ ]:
# Turning every string in 'origin' and 'pastry_type' columns to lowercase and stripping whitespace
df_learn['origin'] = df_learn['origin'].astype(str).str.lower().str.strip()
df_learn['pastry_type'] = df_learn['pastry_type'].astype(str).str.lower().str.strip()

# Solving the problem of different namings for the same pastry
df_learn['pastry_type'] = df_learn['pastry_type'].replace('pastel nata', 'pastel de nata')  
df_learn['origin'] = df_learn['origin'].replace({'nan': np.nan})
df_learn['pastry_type'] = df_learn['pastry_type'].replace({'nan': np.nan})

df_learn.dropna(subset=['quality_class'], inplace=True)
df_learn.drop(columns=['pastry_type'], inplace=True)
df_learn.drop('notes_baker', axis=1, inplace=True)
df_learn.drop_duplicates(inplace=True)
df_learn.reset_index(drop=True, inplace=True)

# Setting these values to NaN instead of removing the rows, to avoid losing too much data

# Sugar content cannot exceed 75g per 100g
df_learn.loc[df_learn['sugar_content'] > 75, 'sugar_content'] = np.nan

# Fat percentage cannot exceed 100%
df_learn.loc[df_learn['cream_fat_content'] > 100, 'cream_fat_content'] = np.nan

# Salt > 100g per kg is inedible 
df_learn.loc[df_learn['salt_ratio'] > 100, 'salt_ratio'] = np.nan

# Eggs cook at ~65C. 170ºC or 575ºC, for example, is impossible for raw egg addition
df_learn.loc[df_learn['egg_temperature'] > 100, 'egg_temperature'] = np.nan

# Oven/Final temp > 500ºC is likely an error (or a mistake using ºF instead of ºC)
df_learn.loc[df_learn['final_temperature'] > 400, 'final_temperature'] = np.nan
df_learn.loc[df_learn['oven_temperature'] > 400, 'oven_temperature'] = np.nan

# Define X (features) and y (target)
# Manually mapping the target variable to ensure '1' is the positive class 'OK'
X = df_learn.drop('quality_class', axis=1)
y = df_learn['quality_class'].map({'OK': 1, 'KO': 0}).astype(int)
X_predict = df_predict.copy()

# Split into training and testing sets, using stratify to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Checking if the dataset is balanced in both sets
print("Training Set Class Distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest Set Class Distribution:")
print(y_test.value_counts(normalize=True))

# --- 1. Identify Column Types ---
# We treat them differently: Numbers get Median, Text gets Mode (Most Frequent)
numerical_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# --- 2. Calculate Statistics (ON TRAIN ONLY) ---
# This prevents data leakage. We learn from Train, and apply to Test.
train_medians = X_train[numerical_cols].median()
train_modes = X_train[categorical_cols].mode().iloc[0]

# --- 3. Impute Missing Values ---
# Fill Numerical
X_train[numerical_cols] = X_train[numerical_cols].fillna(train_medians)
X_test[numerical_cols] = X_test[numerical_cols].fillna(train_medians)

# Fill Categorical
X_train[categorical_cols] = X_train[categorical_cols].fillna(train_modes)
X_test[categorical_cols] = X_test[categorical_cols].fillna(train_modes)


# List of columns that are prone to outliers (Continuous variables)
outlier_cols = ['baking_duration', 'cooling_period', 'sugar_content', 
                'salt_ratio', 'egg_temperature', 'final_temperature', 'oven_temperature',
                'preheating_time', 'vanilla_extract']


for col in outlier_cols:
    # 1. Calculate Limits on TRAINING Data Only (Prevent Leakage)
    lower_limit = X_train[col].quantile(0.01) # 1st Percentile
    upper_limit = X_train[col].quantile(0.99) # 99th Percentile
    
    # 2. Apply to X_train
    # "Clip" is a faster pandas method that does Flooring and Capping in one line
    X_train[col] = X_train[col].clip(lower=lower_limit, upper=upper_limit)
    
    # 3. Apply to X_test (Using Train limits)
    X_test[col] = X_test[col].clip(lower=lower_limit, upper=upper_limit)
 

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# 1. Select categorical columns if you haven't defined this list yet
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Initialize the encoder
# handle_unknown='ignore': If X_test has a category not seen in X_train, it won't crash (it sets all columns to 0).
# sparse_output=False: Returns a regular array/dataframe instead of a compressed matrix.
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 3. Fit on Train, Transform both
# We fit only on X_train to avoid data leakage
train_encoded = ohe.fit_transform(X_train[categorical_cols])
test_encoded = ohe.transform(X_test[categorical_cols])

# 4. Convert back to DataFrames with readable column names
new_columns = ohe.get_feature_names_out(categorical_cols)

X_train_cat = pd.DataFrame(train_encoded, columns=new_columns, index=X_train.index)
X_test_cat = pd.DataFrame(test_encoded, columns=new_columns, index=X_test.index)

# 5. Join with numerical columns and remove original categorical columns
X_train_final = pd.concat([X_train[numerical_cols], X_train_cat], axis=1)
X_test_final = pd.concat([X_test[numerical_cols], X_test_cat], axis=1)

# Check the result
print(f"Original shape: {X_train.shape}")
print(f"Encoded shape:  {X_train_final.shape}")
X_train_final.head()
y_train.head()

Training Set Class Distribution:
quality_class
1    0.635008
0    0.364992
Name: proportion, dtype: float64

Test Set Class Distribution:
quality_class
1    0.635577
0    0.364423
Name: proportion, dtype: float64
Original shape: (4159, 15)
Encoded shape:  (4159, 16)


4142    1
1192    1
908     1
4665    0
4897    0
Name: quality_class, dtype: int64

Feature Selection

In [50]:
cols_to_drop = ['ambient_humidity', 'cream_fat_content','lemon_zest_ph','origin_porto']

X_train_final = X_train_final.drop(columns=cols_to_drop)
X_test_final = X_test_final.drop(columns=cols_to_drop)

Modeling

In [51]:
final_model= AdaBoostClassifier(estimator= RandomForestClassifier(),n_estimators=100, learning_rate=1.0, random_state=42)
final_model.fit(X_train_final, y_train)
y_pred_final = final_model.predict(X_test_final)
print("AdaBoost Classifier 2 Accuracy:", accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

AdaBoost Classifier 2 Accuracy: 0.7913461538461538
              precision    recall  f1-score   support

           0       0.73      0.67      0.70       379
           1       0.82      0.86      0.84       661

    accuracy                           0.79      1040
   macro avg       0.78      0.77      0.77      1040
weighted avg       0.79      0.79      0.79      1040



In [53]:
X_train, X_val, y_train, y_val = train_test_split(X_train_final, y_train, test_size=0.2, random_state=42, stratify=y_train)

print("--- Internal Validation Results ---")
print(classification_report(y_test, y_pred_final))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_final))

# 2. FINAL TRAINING (On Full Dataset)
print("\nRetraining on full dataset...")
final_model.fit(X_train_final, y_train) 

ValueError: Found input variables with inconsistent numbers of samples: [4159, 3327]

Kaggle submission

In [ ]:
# Predict on the unseen data
predictions_encoded = final_model.predict(X_pred_final)

# Decode predictions (0/1 -> KO/OK)
predictions_labels = le.inverse_transform(predictions_encoded)

# Create submission DataFrame
submission = pd.DataFrame({
    'row_id': range(1, len(predictions_labels) + 1), # Matches sampred.csv structure usually
    'quality_class': predictions_labels
})

# Verify format matches sampred.csv (row_id might vary based on the provided sample)
# If sampred.csv has no row_id and just the class, adjust accordingly.
# Based on standard Kaggle:
print(submission.head())

# Save
submission.to_csv('ML08_NB9_submission.csv', index=False)
print("Submission file saved successfully!")